# Tome C3, Chapter 5: The λ-Calculus and Types (Python companion)

Exercises first, solutions at the end. We build a small λ-calculus machine: a parser, β-reduction with de Bruijn indices, reduction strategies and reduction graphs. With it we compute with Church numerals (Kleene's predecessor included), define recursion with the $\mathsf Y$ combinator, test the Church–Rosser theorem on random terms, infer principal types by unification, watch typed terms terminate as Tait's theorem promises, and search for proofs in minimal logic, finding none for Peirce's law.

*Before each solution, predict what the output will look like.*

In [ ]:
import itertools, random, re, time
from collections import deque
import matplotlib.pyplot as plt
import sys
sys.setrecursionlimit(20000)
random.seed(5)

## Exercises

**P5.1 (★) A λ-machine.** Write a parser for terms such as `\x y. x (y y)`, convert them to de Bruijn indices (variables replaced by the number of binders between them and their λ, which makes $\alpha$-equivalence plain equality), implement substitution, one-step reducts and the leftmost (normal-order) strategy. Recompute the reduction graph of Figure 5.1, and check that $\Omega$ reduces to itself.

**P5.2 (★) Church arithmetic.** Implement Church numerals, check Proposition 5.10, Exercise 5.2 (exponentiation), Kleene's predecessor (Proposition 5.11) and the subtraction of Exercise 5.5, and count the normal-order steps each one takes.

**P5.3 (★★) Recursion from nothing.** Define the factorial with $\mathsf Y$ (Theorem 5.12) and Church numerals, and compute $3!$ and $4!$ by normal-order reduction. Check Turing's $\Theta$ (Exercise 5.4).

**P5.4 (★★) Confluence in practice.** Generate random terms, explore their reduction graphs up to a bound, and check that whenever two normal forms are found they are equal (Corollary 5.8), and that any two one-step reducts have a common reduct (Theorem 5.3). Compute the complete development $M^*$ (Definition 5.6) and check that it is reachable from every one-step reduct.

**P5.5 (★) Strategies.** Compare the leftmost strategy with the rightmost-innermost one on $\mathsf K\mathsf I\Omega$, $(\lambda x.y)\Omega$ and on terms that duplicate a costly argument (Exercise 5.6).

**P5.6 (★★) Type inference.** Implement Hindley's algorithm: assign type variables, collect equations, solve them by unification with the occurs check. Find the principal types of $\mathsf I,\mathsf K,\mathsf S$, the numerals, $\mathsf{plus}$, $\mathsf{mult}$ and $\mathsf{pred}$, and check that $\lambda x.xx$, $\Omega$ and $\mathsf Y$ are rejected.

**P5.7 (★★) Termination of typed terms.** Generate random closed terms, keep the typable ones, and compute the length $\nu(M)$ of their longest reduction (finite by Theorem 5.16). How do untyped random terms behave?

**P5.8 (★★) Proof search.** By Lemma 5.21, a normal proof of an implication has a forced shape. Write a search for closed normal terms of a given simple type (an *inhabitation* search), with a loop check, and use it to prove (and refute) formulas of minimal logic: the types of $\mathsf K$ and $\mathsf S$, Exercise 5.9 and Peirce's law.

## Solutions

### P5.1

In [ ]:
def parse(s):
    toks = re.findall(r"\\|λ|\.|\(|\)|[A-Za-z0-9_']+", s)
    pos = 0
    def peek():
        return toks[pos] if pos < len(toks) else None
    def eat():
        nonlocal pos
        pos += 1
        return toks[pos - 1]
    def term():
        if peek() in ('\\', 'λ'):
            eat()
            names = []
            while peek() != '.':
                names.append(eat())
            eat()
            body = term()
            for x in reversed(names):
                body = ('lam', x, body)
            return body
        t = atom()
        while peek() not in (None, ')'):
            if peek() in ('\\', 'λ'):
                return ('app', t, term())
            t = ('app', t, atom())
        return t
    def atom():
        if peek() == '(':
            eat()
            t = term()
            eat()
            return t
        return ('var', eat())
    return term()

def to_db(t, env=()):
    # named term -> de Bruijn: ('v', k) | ('v', name) for free | ('l', body) | ('a', f, x)
    if t[0] == 'var':
        return ('v', env.index(t[1])) if t[1] in env else ('f', t[1])
    if t[0] == 'lam':
        return ('l', to_db(t[2], (t[1],) + env))
    return ('a', to_db(t[1], env), to_db(t[2], env))

def shift(t, d, c=0):
    if t[0] == 'v':
        return ('v', t[1] + d) if t[1] >= c else t
    if t[0] == 'l':
        return ('l', shift(t[1], d, c + 1))
    if t[0] == 'a':
        return ('a', shift(t[1], d, c), shift(t[2], d, c))
    return t

def subst(t, j, s):
    if t[0] == 'v':
        return s if t[1] == j else t
    if t[0] == 'l':
        return ('l', subst(t[1], j + 1, shift(s, 1)))
    if t[0] == 'a':
        return ('a', subst(t[1], j, s), subst(t[2], j, s))
    return t

def contract(redex):
    (_, (_, body), arg) = redex
    return shift(subst(body, 0, shift(arg, 1)), -1)

def reducts(t):
    out = []
    if t[0] == 'a':
        if t[1][0] == 'l':
            out.append(contract(t))
        out += [('a', f, t[2]) for f in reducts(t[1])]
        out += [('a', t[1], x) for x in reducts(t[2])]
    elif t[0] == 'l':
        out += [('l', b) for b in reducts(t[1])]
    return out

def leftmost(t):
    # one leftmost-outermost step, or None if t is normal
    if t[0] == 'a':
        if t[1][0] == 'l':
            return contract(t)
        f = leftmost(t[1])
        if f is not None:
            return ('a', f, t[2])
        x = leftmost(t[2])
        return None if x is None else ('a', t[1], x)
    if t[0] == 'l':
        b = leftmost(t[1])
        return None if b is None else ('l', b)
    return None

def normalise(t, limit=10**5, step=leftmost):
    for k in range(limit):
        t2 = step(t)
        if t2 is None:
            return t, k
        t = t2
    return None, limit

def show(t, names='xyzwuvabcdeghijk'):
    def go(t, env, depth):
        if t[0] == 'v':
            return env[t[1]]
        if t[0] == 'f':
            return t[1]
        if t[0] == 'l':
            x = names[depth % len(names)] + ("'" * (depth // len(names)))
            return 'λ' + x + '.' + go(t[1], (x,) + env, depth + 1)
        f = go(t[1], env, depth)
        if t[1][0] == 'l':
            f = '(' + f + ')'
        x = go(t[2], env, depth)
        if t[2][0] in ('a', 'l'):
            x = '(' + x + ')'
        return f + ' ' + x
    return go(t, (), 0)

L = lambda s: to_db(parse(s))
I, K, S = L(r"\x.x"), L(r"\x y.x"), L(r"\x y z. x z (y z)")
OMEGA = L(r"(\x. x x) (\x. x x)")
print("Omega -> ", [show(r) for r in reducts(OMEGA)], " equal to Omega:", reducts(OMEGA) == [OMEGA])

def reduction_graph(t, max_nodes=10**4):
    seen, edges, queue = {t}, [], deque([t])
    while queue and len(seen) < max_nodes:
        u = queue.popleft()
        for v in reducts(u):
            edges.append((u, v))
            if v not in seen:
                seen.add(v)
                queue.append(v)
    return seen, edges

nodes, edges = reduction_graph(L(r"(\x. x x) ((\y.y) (\y.y))"))
print(len(nodes), "terms and", len(edges), "arrows, as in Figure 5.1; normal forms:",
      [show(n) for n in nodes if not reducts(n)])

De Bruijn indices make the machine short: no renaming is ever needed, and two terms are $\alpha$-equivalent exactly when their index forms are equal. The graph of Figure 5.1 has $7$ terms and $9$ arrows, with the single normal form $\lambda x.x$.

### P5.2

In [ ]:
def church(n):
    body = ('v', 0)
    for _ in range(n):
        body = ('a', ('v', 1), body)
    return ('l', ('l', body))

def unchurch(t):
    if t is None or t[0] != 'l' or t[1][0] != 'l':
        return None
    b, n = t[1][1], 0
    while b[0] == 'a' and b[1] == ('v', 1):
        b, n = b[2], n + 1
    return n if b == ('v', 0) else None

def app(*ts):
    t = ts[0]
    for u in ts[1:]:
        t = ('a', t, u)
    return t

D = {}
D['succ'] = L(r"\n f x. f (n f x)")
D['plus'] = L(r"\m n f x. m f (n f x)")
D['mult'] = L(r"\m n f. m (n f)")
D['true'], D['false'] = L(r"\x y.x"), L(r"\x y.y")
D['pair'] = L(r"\a b z. z a b")
D['fst'], D['snd'] = L(r"\p. p (\x y.x)"), L(r"\p. p (\x y.y)")
D['iszero'] = app(L(r"\F T n. n (\z. F) T"), D['false'], D['true'])
D['step'] = app(L(r"\P S U. \p. P (S p) (U (S p))"), D['pair'], D['snd'], D['succ'])
D['pred'] = app(L(r"\F St P z. \n. F (n St (P z z))"), D['fst'], D['step'], D['pair'], church(0))
D['sub'] = app(L(r"\pred. \m n. n pred m"), D['pred'])

def run(t):
    nf, steps = normalise(t)
    return unchurch(nf), steps

checks = {
    'succ': all(run(app(D['succ'], church(n)))[0] == n + 1 for n in range(8)),
    'plus': all(run(app(D['plus'], church(m), church(n)))[0] == m + n for m in range(6) for n in range(6)),
    'mult': all(run(app(D['mult'], church(m), church(n)))[0] == m * n for m in range(6) for n in range(6)),
    'exp n m = m^n': all(run(app(church(n), church(m)))[0] == m**n for m in range(4) for n in range(1, 4)),
    'pred': all(run(app(D['pred'], church(n)))[0] == max(n - 1, 0) for n in range(10)),
    'sub': all(run(app(D['sub'], church(m), church(n)))[0] == max(m - n, 0) for m in range(6) for n in range(6)),
}
print(checks)
print("normal-order steps of pred on n = 0..12:", [run(app(D['pred'], church(n)))[1] for n in range(13)])
print("steps of mult on n x n, n = 1..8:", [run(app(D['mult'], church(n), church(n)))[1] for n in range(1, 9)])

All the encodings work. Kleene's predecessor costs a number of steps linear in $n$: it rebuilds the number from $0$ to get one step of history. (Church's own doubts were reasonable: with numerals that are iterators, there is no way to ``peel off'' one application in constant time; other encodings, such as Scott's, have a constant-time predecessor and no built-in iteration.)

### P5.3

In [ ]:
D['Y'] = L(r"\f. (\x. f (x x)) (\x. f (x x))")
D['Theta'] = app(L(r"\x y. y (x x y)"), L(r"\x y. y (x x y)"))
# fact = Y (\f n. iszero n 1 (mult n (f (pred n))))
FACT_BODY = app(L(r"\IZ M P one. \f n. IZ n one (M n (f (P n)))"), D['iszero'], D['mult'], D['pred'], church(1))
FACT = app(D['Y'], FACT_BODY)
for n in range(4):
    t0 = time.time()
    v, steps = run(app(FACT, church(n)))
    print(f"fact {n} = {v}  ({steps} normal-order steps, {time.time() - t0:.2f} s)")

F = ('f', 'F')
t = app(D['Theta'], F)
t1 = leftmost(leftmost(t))
print("Theta F ->> F (Theta F):", t1 == app(F, app(D['Theta'], F)))

Recursion works without any name for the function being defined: $\mathsf Y$ hands the body a copy of itself. (Computing $4!$ the same way takes $14283$ steps, about five seconds.) Normal order matters: under the rightmost-innermost strategy, $\mathsf Y G$ would unfold forever before anything else happens. The step counts grow fast, since numbers are unary and the multiplication is done by iterating iterators.

### P5.4

In [ ]:
def random_term(size, depth=0):
    # a random closed term with roughly `size` nodes
    if size <= 1:
        return ('v', random.randrange(depth)) if depth else ('l', ('v', 0))
    r = random.random()
    if r < 0.35 or depth == 0:
        return ('l', random_term(size - 1, depth + 1))
    k = random.randint(1, size - 2) if size > 2 else 1
    return ('a', random_term(k, depth), random_term(size - 1 - k, depth))

def complete_development(t):
    if t[0] == 'l':
        return ('l', complete_development(t[1]))
    if t[0] == 'a':
        if t[1][0] == 'l':
            return contract(('a', ('l', complete_development(t[1][1])), complete_development(t[2])))
        return ('a', complete_development(t[1]), complete_development(t[2]))
    return t

def reach(t, limit=3000):
    seen, queue = {t}, deque([t])
    while queue and len(seen) < limit:
        for v in reducts(queue.popleft()):
            if v not in seen:
                seen.add(v)
                queue.append(v)
    return seen

tested = nf_ok = diamond_ok = dev_ok = 0
for trial in range(400):
    M = random_term(random.randint(6, 16))
    graph = reach(M)
    nfs = {u for u in graph if not reducts(u)}
    if len(graph) >= 3000:
        continue
    tested += 1
    nf_ok += len(nfs) <= 1
    R1 = list(dict.fromkeys(reducts(M)))
    Ms = complete_development(M)
    sets = {r: reach(r, 3000) for r in R1}
    diamond_ok += all(sets[a] & sets[b] for a in R1 for b in R1)
    dev_ok += all(Ms in sets[r] for r in R1)
print(f"{tested} random terms with finite explored graphs: at most one normal form in {nf_ok},"
      f" one-step reducts joinable in {diamond_ok}, M* reachable from every reduct in {dev_ok}")

Every test passes, as Theorem 5.3 and Lemma 5.7 predict: $M\to N$ implies $N\Rightarrow M^*$, hence $N\twoheadrightarrow M^*$, so the complete development is a common reduct of all one-step reducts. (Terms whose reduction graph is too large, or infinite, are skipped: the search cannot prove anything about them.)

### P5.5

In [ ]:
def rightmost_innermost(t):
    # contract a redex none of whose subterms is a redex, the rightmost one
    if t[0] == 'a':
        x = rightmost_innermost(t[2])
        if x is not None:
            return ('a', t[1], x)
        f = rightmost_innermost(t[1])
        if f is not None:
            return ('a', f, t[2])
        if t[1][0] == 'l':
            return contract(t)
        return None
    if t[0] == 'l':
        b = rightmost_innermost(t[1])
        return None if b is None else ('l', b)
    return None

tests = {"K I Ω": app(K, I, OMEGA), "(λx.y) Ω": app(L(r"\x.y"), OMEGA),
         "(λx.x+x+x)(3×3)": app(L(r"\P. \x. P x (P x x)"), D['plus'], app(D['mult'], church(3), church(3))),
         "(λx.λy.y)(Y I)": app(L(r"\x y. y"), app(D['Y'], I))}
for name, t in tests.items():
    a = normalise(t, limit=300)
    b = normalise(t, limit=300, step=rightmost_innermost)
    fmt = lambda r: "none within 300 steps" if r[0] is None else f"{r[1]} steps"
    print(f"{name:22s} leftmost: {fmt(a):34s} rightmost-innermost: {fmt(b)}")

The leftmost strategy finds every normal form that exists (the normalisation theorem), but it may repeat work: it substitutes the unevaluated argument $3\times3$ three times and then evaluates each copy. The innermost strategy evaluates the argument once, which is what most programming languages do, but it loops on $\Omega$ and on the fixed-point combinator even when the result does not need them. Lazy languages such as Haskell use the leftmost strategy with *sharing*: the argument is evaluated at most once, when first needed.

### P5.6

In [ ]:
class TypeError_(Exception):
    pass

def infer(t):
    # Hindley's algorithm on de Bruijn terms; types: int (variable) or ('->', A, B).
    counter = itertools.count()
    sub = {}
    def find(a):
        while isinstance(a, int) and a in sub:
            a = sub[a]
        return a
    def occurs(v, a):
        a = find(a)
        if isinstance(a, int):
            return a == v
        return occurs(v, a[1]) or occurs(v, a[2])
    def unify(a, b):
        a, b = find(a), find(b)
        if a == b:
            return
        if isinstance(a, int):
            if occurs(a, b):
                raise TypeError_("occurs check: a type would contain itself")
            sub[a] = b
        elif isinstance(b, int):
            unify(b, a)
        else:
            unify(a[1], b[1])
            unify(a[2], b[2])
    def go(t, env):
        if t[0] == 'v':
            return env[t[1]]
        if t[0] == 'f':
            return next(counter)
        if t[0] == 'l':
            a = next(counter)
            return ('->', a, go(t[1], (a,) + env))
        f, x, r = go(t[1], env), go(t[2], env), next(counter)
        unify(f, ('->', x, r))
        return r
    ty = go(t, ())
    def resolve(a):
        a = find(a)
        return a if isinstance(a, int) else ('->', resolve(a[1]), resolve(a[2]))
    return resolve(ty)

def show_type(a):
    names = {}
    def go(a, left):
        if isinstance(a, int):
            names.setdefault(a, "ABCDEFGHJKLMNPQRSTUVW"[len(names)] if len(names) < 21 else f"T{len(names)}")
            return names[a]
        s = go(a[1], True) + "→" + go(a[2], False)
        return "(" + s + ")" if left else s
    return go(a, False)

def unifiable(a, b):
    # do the types a and b (with separate variables) have a common instance?
    sub = {}
    def ren(t, off):
        return t + off if isinstance(t, int) else (t if isinstance(t, str) else ('->', ren(t[1], off), ren(t[2], off)))
    def find(x):
        while isinstance(x, int) and x in sub:
            x = sub[x]
        return x
    def occurs(v, x):
        x = find(x)
        return x == v if not isinstance(x, tuple) else occurs(v, x[1]) or occurs(v, x[2])
    def unify(x, y):
        x, y = find(x), find(y)
        if x == y:
            return True
        if isinstance(x, int):
            if occurs(x, y):
                return False
            sub[x] = y
            return True
        if isinstance(y, int):
            return unify(y, x)
        if isinstance(x, str) or isinstance(y, str):
            return False
        return unify(x[1], y[1]) and unify(x[2], y[2])
    return unify(ren(a, 0), ren(b, 10**6))

Nat = lambda X: ('->', ('->', X, X), ('->', X, X))
pt = infer(D['pred'])
print("pred : N_X -> N_X for one type X?", unifiable(pt, ('->', Nat(0), Nat(0))))
print("pred : N_Y -> N_X for some types X, Y?", unifiable(pt, ('->', Nat(1), Nat(0))))
print("plus : N_X -> N_X -> N_X?", unifiable(infer(D['plus']), ('->', Nat(0), ('->', Nat(0), Nat(0)))))
for name, t in [("I", I), ("K", K), ("S", S), ("numeral 2", church(2)), ("plus", D['plus']),
                ("mult", D['mult']), ("pred", D['pred']), ("λx.x x", L(r"\x. x x")), ("Ω", OMEGA), ("Y", D['Y'])]:
    try:
        print(f"{name:10s} : {show_type(infer(t))}")
    except TypeError_ as e:
        print(f"{name:10s} : not typable ({e})")

The principal types come out as expected: $\mathsf S:(A\to B\to C)\to(A\to B)\to A\to C$, numerals $(A\to A)\to A\to A$, and $\mathsf{plus}$ has a type more general than $N_A\to N_A\to N_A$ (Exercise 5.8). The principal type of $\mathsf{pred}$ is enormous, and unification shows what it means: $\mathsf{pred}$ has a type $N_Y\to N_X$, its argument being a numeral at a type $Y$ of pairs, but no type $N_X\to N_X$. In the simply typed calculus the predecessor cannot be applied to its own result at the same type, one of the weaknesses that Schwichtenberg's theorem makes precise. Self-application fails the occurs check, which is the argument of Exercise 5.7 performed by a machine.

### P5.7

In [ ]:
def nu(t, limit=20000):
    # longest reduction length, by memoised search on the (finite, acyclic) reduction graph
    memo = {}
    count = [0]
    def go(u):
        if u in memo:
            return memo[u]
        count[0] += 1
        if count[0] > limit:
            raise RecursionError
        memo[u] = 1 + max((go(v) for v in reducts(u)), default=-1)
        return memo[u]
    return go(t)

typed, untyped_stats = [], {'loop or too big': 0, 'finite': 0}
for trial in range(3000):
    t = random_term(random.randint(5, 14))
    try:
        infer(t)
        ok = True
    except TypeError_:
        ok = False
    try:
        n = nu(t)
    except RecursionError:
        n = None
    if ok:
        typed.append(n)
    else:
        untyped_stats['finite' if n is not None else 'loop or too big'] += 1
print(f"typable random terms: {len(typed)}, all with finite longest reduction: {all(n is not None for n in typed)};"
      f" longest: {max(typed)}")
print("untypable random terms:", untyped_stats)
plt.figure(figsize=(6, 3.2))
plt.hist(typed, bins=range(0, max(typed) + 2), color='tab:blue')
plt.xlabel(r'$\nu(M)$, longest reduction'); plt.ylabel('typable terms'); plt.title('Typed terms terminate'); plt.show()

Every typable term has a finite reduction graph, so its longest reduction $\nu(M)$ exists, as Theorem 5.16 asserts; among untypable terms, some loop (or explode beyond the search limit). Typability is sufficient for termination, not necessary: many untypable terms still terminate, such as $(\lambda x.xx)\,\mathsf I$.

### P5.8

In [ ]:
def arrow(*ts):
    t = ts[-1]
    for a in reversed(ts[:-1]):
        t = ('->', a, t)
    return t

def inhabit(goal, ctx=(), stack=frozenset()):
    # Search a normal term of type `goal` in context ctx (list of (name, type)); atoms are strings.
    if isinstance(goal, tuple):
        x = f"x{len(ctx)}"
        body = inhabit(goal[2], ctx + ((x, goal[1]),), stack)
        return None if body is None else f"λ{x}.{body}"
    key = (frozenset(ty for _, ty in ctx), goal)
    if key in stack:
        return None                         # same goal with the same hypotheses: a loop
    stack = stack | {key}
    for name, ty in reversed(ctx):
        args, t = [], ty
        while isinstance(t, tuple):
            args.append(t[1])
            t = t[2]
        for k in range(len(args) + 1):
            res = ty
            for _ in range(k):
                res = res[2]
            if res == goal:
                subterms = []
                for a in args[:k]:
                    m = inhabit(a, ctx, stack)
                    if m is None:
                        break
                    subterms.append(m if not ' ' in m and not m.startswith('λ') else '(' + m + ')')
                else:
                    return name + ''.join(' ' + m for m in subterms)
    return None

p, q, r = 'p', 'q', 'r'
for name, ty in [("K: p→q→p", arrow(p, q, p)),
                 ("S: (p→q→r)→(p→q)→p→r", arrow(arrow(p, q, r), arrow(p, q), p, r)),
                 ("((p→q)→p)→(p→q)→q", arrow(arrow(arrow(p, q), p), arrow(p, q), q)),
                 ("Peirce ((p→q)→p)→p", arrow(arrow(arrow(p, q), p), p)),
                 ("p→q", arrow(p, q)),
                 ("double negation intro p→(p→q)→q", arrow(p, arrow(p, q), q))]:
    print(f"{name:36s} {inhabit(ty) or 'no proof (no closed normal term)'}")

The search returns the expected terms ($\lambda x_0x_1x_2.\,x_0x_2(x_1x_2)$ for $\mathsf S$) and finds no proof of Peirce's law. The loop check is sound because a normal proof needing the same goal from the same set of hypotheses twice on one branch can be shortened; with it, the search always terminates, since there are finitely many sets of subformulas. So provability in minimal implicational logic is decidable, through normal forms: Theorem 5.16 turned an infinite search into a finite one.